In [2]:
def posterior_num(prior,evidence,likelihood):
    """
    Docstring for posterior_num
    以数字传入一下数值并计算后验概率
    :param prior: 先验概率
    :param evidence: 证据概率
    :param likelihood: 似然概率
    """
    return (likelihood * prior)/evidence

print(posterior_num(0.7, 0.31, 0.4))

0.9032258064516128


问题： 在训练朴素贝叶斯文本分类器时，我们统计了词汇表中每个词在“体育”类和“科技”类新闻中出现的次数。现在要对一篇新文章分类，文章里出现了“算法”和“篮球”两个词。已知“篮球”在体育类中出现的概率远高于科技类。如果朴素贝叶斯模型最终将文章分类为“科技”，最可能的原因是什么？

A. 文章很短，所以特征少。
B. “算法”这个词在科技类中出现的概率极高，其影响力压过了“篮球”。
C. 先验概率 P(体育) 设置得太低了。
D. “篮球”和“算法”这两个词在现实中是相关的，违反了独立性假设。

在朴素贝叶斯的实际决策中，特征的条件概率（似然）往往拥有更大的权重和影响力。

“算法”是一个科技领域的强相关词，它在科技类文章中出现的概率可能极高（例如，接近1），而在体育类文章中出现的概率极低（例如，接近0）。这种巨大的差异会产生一个极强的证据，支持科技类。
“篮球”是体育领域的强相关词，但它也可能偶尔出现在科技文章（比如讨论体育科技的论文）中。所以，虽然 P(篮球|体育) >> P(篮球|科技)，但其差异幅度可能不如“算法”这个词的差异那么极端。
因此，更常见、更直接的情景是： “算法”这个词为“科技”类提供的证据强度如此之大，以至于完全压倒了“篮球”为“体育”类提供的证据。即使两类先验概率相等，最终也是科技类得分更高。

所以，在给定的选项中，B选项是更精准、更本质的解释。它直接体现了特征（词）本身判别力的强弱如何主导分类结果。

让我们用一个小计算来感受一下：
假设一些虚拟但合理的数据：

P(科技) = 0.5, P(体育) = 0.5 （先验相等，排除C选项的影响）
P(算法|科技) = 0.9, P(算法|体育) = 0.01
P(篮球|体育) = 0.7, P(篮球|科技) = 0.05
计算得分：

科技得分 = 0.5 * 0.9 * 0.05 = 0.0225
体育得分 = 0.5 * 0.01 * 0.7 = 0.0035
看，即使“篮球”强烈指向体育（0.7 vs 0.05），但因为“算法”强烈指向科技（0.9 vs 0.01）且幅度更大，最终模型仍然将文章分类为科技。这就是B选项描述的情况。

======================================================================
======================================================================

数学模型：
我们想求的是在已知水果特征（颜色=红，形状=圆）的条件下，它属于某个类别（苹果或橘子）的概率。用数学公式表示就是求 P(类别 | 特征)。

根据贝叶斯定理，这个公式可以转化为：
P(类别 | 特征) = [ P(特征 | 类别) * P(类别) ] / P(特征)

对于分类任务，我们不需要精确的概率值，只需要比较不同类别的这个值哪个更大。由于分母P(特征)对所有类别都一样，所以我们实际上只需要比较分子：
P(类别 | 特征) ∝ P(特征 | 类别) * P(类别)

P(类别)：这叫先验概率。比如，在你的水果篮里，如果苹果比橘子多得多，那么即使看到一个圆形水果，你也会先入为主地更倾向于猜它是苹果。这个概率可以直接从数据中统计（苹果数量/总水果数）。
P(特征 | 类别)：这叫条件概率或似然。意思是，在已知是苹果的条件下，看到它是红色的概率有多大？在已知是橘子的条件下，看到它是红色的概率又有多大？这是我们模型需要从数据中学的关键部分。

第二步：最大似然估计参数  
核心思想就是：频率 = 概率。 我们直接用数据中出现的频率来作为概率的最佳估计

## 项目：基于PyTorch的朴素贝叶斯情感分析器
项目目标
使用朴素贝叶斯算法，训练一个能自动判断电影评论是正面（positive）还是负面（negative）的分类器。

In [3]:
# 训练数据：每条是一个(评论文本, 标签)的元组
# 标签：1表示正面，0表示负面
train_data = [
    ("这部电影太精彩了，演员演技炸裂", 1),
    ("完美的剧情，令人难忘的结局", 1),
    ("特效震撼，音乐动人，强烈推荐", 1),
    ("导演功力深厚，画面美轮美奂", 1),
    ("无聊透顶，浪费时间", 0),
    ("演技尴尬，剧情老套", 0),
    ("最烂的电影，没有之一", 0),
    ("毫无逻辑，看得我想睡觉", 0)
]


### 我的代码（无拉普拉斯平滑）

In [4]:
# 数据处理

def preprocess_text(text):
    """删除标点"""
    punctuations = "，。“”‘’|、《》——？/"
    for punc in punctuations:
        text = text.replace(punc, '')
    
    return text

def token(text):
    """分割语句，储存到列表"""
    tokens = []
    for word in text:
        tokens.append(word)
    return tokens

def reload_verctors(tokens, vectors):
    """更新数据向量"""
    for word in tokens:
        if word in vectors:
            vectors[word] += 1
        if not word in vectors:
            vectors[word] = 1
    return vectors

vectors_1 = {}  # 正面评论
vectors_0 = {}  # 负面评论

for data in train_data:
    tokens = token(preprocess_text(data[0]))
    if data[1] == 0:
        reload_verctors(tokens, vectors_0)
    else:
        reload_verctors(tokens, vectors_1)

In [5]:
def NaiveBayes(tokens, vectors_0, vectors_1):
    P_0,P_1 = 0, 0
    for words in tokens:
        if words in vectors_0:
            x = (vectors_0[words]/len(vectors_0))*0.5/(len(vectors_0)+len(vectors_1))
            P_0 += x
        elif words in vectors_1:
            y = (vectors_1[words]/len(vectors_1))*0.5/(len(vectors_0)+len(vectors_1))
            P_1 += y
    if P_0 < P_1:
        return "好评"
    else:
        return "差评"


comment = input("请输入你对这部电影的评价")
tokens = token(preprocess_text(comment))
print(NaiveBayes(tokens, vectors_0, vectors_1))

好评


# 标准答案（有拉普拉斯平滑）

In [6]:
def build_vocab(train_data):
    """构建所有词语的词汇表"""
    vocab = {}
    for text, label in train_data:
        cleaned = preprocess_text(text)
        tokens = token(cleaned)
        for word in tokens:
            if word not in vocab:
                vocab[word] = len(vocab)  # 给每个词分配唯一ID
    return vocab

# 使用
vocab = build_vocab(train_data)
print(f"词汇表大小: {len(vocab)}")
print(f"词汇表示例: {list(vocab.items())[:5]}")

词汇表大小: 71
词汇表示例: [('这', 0), ('部', 1), ('电', 2), ('影', 3), ('太', 4)]


In [7]:
def compute_class_stats(train_data, vocab, alpha=1.0):
    """
    计算每个类别的统计信息
    返回: class_priors, word_probs_0, word_probs_1
    """
    # 初始化计数
    total_docs = len(train_data)
    docs_in_class_0 = sum(1 for _, label in train_data if label == 0)
    docs_in_class_1 = total_docs - docs_in_class_0
    
    # 先验概率
    prior_0 = docs_in_class_0 / total_docs
    prior_1 = docs_in_class_1 / total_docs
    
    # 词语计数初始化
    word_count_0 = {word: 0 for word in vocab}
    word_count_1 = {word: 0 for word in vocab}
    total_words_0 = 0
    total_words_1 = 0
    
    # 统计每个类别的词频
    for text, label in train_data:
        cleaned = preprocess_text(text)
        tokens = token(cleaned)
        for word in tokens:
            if word in vocab:  # 确保词在词汇表中
                if label == 0:
                    word_count_0[word] += 1
                    total_words_0 += 1
                else:
                    word_count_1[word] += 1
                    total_words_1 += 1
    
    # 计算条件概率（应用拉普拉斯平滑）
    V = len(vocab)  # 词汇表大小
    word_prob_0 = {}
    word_prob_1 = {}
    
    for word in vocab:
        # P(word|class_0) = (count(word, class_0) + alpha) / (total_words_0 + alpha * V)
        word_prob_0[word] = (word_count_0[word] + alpha) / (total_words_0 + alpha * V)
        word_prob_1[word] = (word_count_1[word] + alpha) / (total_words_1 + alpha * V)
    
    return prior_0, prior_1, word_prob_0, word_prob_1

In [8]:
def NaiveBayes_corrected(text, vocab, prior_0, prior_1, word_prob_0, word_prob_1):
    """修正后的朴素贝叶斯预测"""
    # 预处理文本
    cleaned = preprocess_text(text)
    tokens = token(cleaned)
    
    # 初始化对数概率（使用对数防止数值下溢）
    log_prob_0 = np.log(prior_0)  # 需要 import numpy as np
    log_prob_1 = np.log(prior_1)
    
    # 对每个词语累加对数概率
    for word in tokens:
        if word in vocab:  # 只考虑词汇表中的词
            log_prob_0 += np.log(word_prob_0[word])
            log_prob_1 += np.log(word_prob_1[word])
        else:
            # 对于新词，使用均匀分布（拉普拉斯平滑的效果）
            # 或者可以忽略，因为对所有类别影响相同
            pass
    
    # 比较概率
    if log_prob_1 > log_prob_0:
        return "正面评价", log_prob_1, log_prob_0
    else:
        return "负面评价", log_prob_1, log_prob_0

In [9]:
import numpy as np

# 你的原始函数（保持不动）
def preprocess_text(text):
    """删除标点"""
    punctuations = "，。！？；：”“‘’、"
    for punc in punctuations:
        text = text.replace(punc, '')
    return text

def token(text):
    """分割语句，储存到列表"""
    return [char for char in text if char != ' ']



# 1. 构建词汇表
vocab = {}
for text, label in train_data:
    tokens = token(preprocess_text(text))
    for word in tokens:
        if word not in vocab:
            vocab[word] = len(vocab)

print(f"词汇表大小: {len(vocab)}")

# 2. 训练朴素贝叶斯
prior_0, prior_1, word_prob_0, word_prob_1 = compute_class_stats(train_data, vocab, alpha=1.0)

# 3. 测试
test_comments = [
    "这部电影还不错",
    "太烂了，千万别看",
    "演员演技很好，但剧情一般"
]

for comment in test_comments:
    result, prob1, prob0 = NaiveBayes_corrected(comment, vocab, prior_0, prior_1, word_prob_0, word_prob_1)
    print(f"评论: '{comment}'")
    print(f"预测: {result} (正面概率: {np.exp(prob1):.4f}, 负面概率: {np.exp(prob0):.4f})")
    print()


词汇表大小: 71
评论: '这部电影还不错'
预测: 正面评价 (正面概率: 0.0000, 负面概率: 0.0000)

评论: '太烂了，千万别看'
预测: 负面评价 (正面概率: 0.0000, 负面概率: 0.0000)

评论: '演员演技很好，但剧情一般'
预测: 正面评价 (正面概率: 0.0000, 负面概率: 0.0000)

